# **HospitalityGPT - Fine-Tuning TinyLlama using LoRA**

## **Project Objective**

The objective of this notebook is to fine-tune the TinyLlama 1.1B Chat model on the prepared HospitalityGPT dataset using Low-Rank Adaptation (LoRA).

Fine-tuning enables the model to learn hospitality-specific customer support conversations while requiring significantly fewer trainable parameters than full model training.

By combining parameter-efficient fine-tuning with 4-bit quantization, the model can be trained efficiently on limited GPU resources such as Google Colab T4 GPUs.

### **Environment Setup**

The required libraries for model loading, parameter-efficient fine-tuning, quantization, and supervised instruction tuning are installed.

The selected library versions are compatible with each other and provide stable training performance on Google Colab.

In [2]:
# Remove conflicting package from Notebook 1 - Data_Preparation
!pip uninstall -y sentence-transformers

# Install compatible versions
!pip install -q transformers==4.38.2
!pip install -q datasets
!pip install -q trl==0.8.6
!pip install -q peft==0.8.2
!pip install -q accelerate==0.27.2
!pip install -q bitsandbytes
!pip install -q sentencepiece

Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1


**Verify GPU Environment**

Before loading the model, verify that GPU acceleration is available and inspect the CUDA environment.

Since TinyLlama is fine-tuned using 4-bit quantization and LoRA, GPU availability is essential for efficient training.

In [3]:
!nvidia-smi

Sun Jun 28 10:36:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")

PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU Device      : Tesla T4


**Mount Google Drive**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Required Libraries**

In [6]:
import torch
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

In [7]:
import transformers
import trl
import peft

print("Transformers :", transformers.__version__)
print("TRL          :", trl.__version__)
print("PEFT         :", peft.__version__)

Transformers : 4.38.2
TRL          : 0.8.6
PEFT         : 0.8.2


**Configure Project Paths**

In [8]:
BASE_PATH = "/content/drive/MyDrive/Module_6_Specialization_NLP/HospitalityGPT"

DATA_PATH = f"{BASE_PATH}/data"
MODEL_PATH = f"{BASE_PATH}/models"

**Load the Training and Testing Datasets**

In [9]:
train_df = pd.read_csv(f"{DATA_PATH}/train_data.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test_data.csv")

print("Training Dataset :", train_df.shape)
print("Testing Dataset  :", test_df.shape)

train_df.head()

Training Dataset : (20184, 5)
Testing Dataset  : (5046, 5)


,instruction,input,output,intent,category
0,id like to know more about the check-out could...,NaN,If you require further guidance on how to requ...,check_out,CHECK_OUT
1,"I need my fucking rom, how do I check it?",NaN,"To verify your hotel reservation, please adher...",check_hotel_reservation,HOTEL
2,would it be possible to speak with client serv...,NaN,To reach out to our customer service departmen...,customer_service,CONTACT
3,im looking for the fucking hotel offesr i need...,NaN,To provide you with the necessary guidance for...,check_hotel_offers,HOTEL
4,I would lik to know about the fucking check-in...,NaN,I am able to guide you through the process for...,check_in,CHECK_IN


### **Convert the Dataset into Instruction Format**

Large Language Models are trained using instruction-response pairs.

Each training example is therefore converted into a structured prompt containing:

- Instruction
- Input
- Response

This formatting closely matches the instruction tuning process used by many modern conversational language models.

In [10]:
SYSTEM_PROMPT = (
    "You are HospitalityGPT, an AI assistant specialized in hotel and hospitality customer support."
)

def format_prompt(row):

    return f"""{SYSTEM_PROMPT}

### Instruction:
{row['instruction']}

### Input:
{row['input']}

### Response:
{row['output']}"""

In [11]:
train_df["text"] = train_df.apply(format_prompt, axis=1)
test_df["text"] = test_df.apply(format_prompt, axis=1)

train_df["text"].iloc[0]

'You are HospitalityGPT, an AI assistant specialized in hotel and hospitality customer support.\n\n### Instruction:\nid like to know more about the check-out could ya find some information\n\n### Input:\nnan\n\n### Response:\nIf you require further guidance on how to request a late check-out, please adhere to the following steps: 1. Access your account at {{WEBSITE_URL}}. 2. Proceed to the reservations area to review your booking information. 3. Seek the option to adjust or prolong your accommodation, usually located under {{CHECK_OUT_OPTION}}. 4. Choose the late check-out request and indicate your desired time. 5. Finalize your request and await confirmation, which will be sent to you via email or through app notifications. Should you face any complications or require prompt assistance, do not hesitate to reach out to the front desk or customer service for immediate support.'

In [12]:
train_dataset = Dataset.from_pandas(train_df[["text"]])
test_dataset = Dataset.from_pandas(test_df[["text"]])

print(train_dataset)

Dataset({
    features: ['text'],
    num_rows: 20184
})


### **Load the Pre-trained TinyLlama Model**

The TinyLlama 1.1B Chat model is loaded using 4-bit quantization to reduce GPU memory consumption.

Quantization enables efficient fine-tuning on Google Colab T4 GPUs while maintaining strong conversational performance.

NormalFloat 4 (NF4) quantization is used because it is specifically designed for QLoRA training and provides an effective balance between memory efficiency and model accuracy.

In [13]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### **Configure Low-Rank Adaptation (LoRA)**

Instead of updating all model parameters, Low-Rank Adaptation (LoRA) fine-tunes only a small number of additional trainable parameters.

This significantly reduces memory consumption and training time while preserving the knowledge learned during pre-training.

LoRA is particularly effective for domain adaptation tasks such as hospitality customer support.

In [17]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [18]:
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


### **Configure Training Parameters**

The training configuration defines the hyperparameters used during supervised fine-tuning.

The selected values are chosen to balance training efficiency, GPU memory usage, and model performance on Google Colab's T4 GPU.

Parameter-efficient fine-tuning with LoRA allows the model to learn hospitality-specific knowledge while updating only a small subset of trainable parameters.

In [19]:
training_arguments = TrainingArguments(

    output_dir=f"{MODEL_PATH}/training_output",

    num_train_epochs=4,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    weight_decay=0.01,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    logging_steps=25,

    save_strategy="epoch",

    evaluation_strategy="no",

    fp16=True,

    report_to="none"

)

**Why these values?**

These are well-balanced for this project:

* 4 epochs → sufficient learning without unnecessary overfitting.
* Effective batch size = 16 (4 x gradient accumulation of 4).
* Cosine scheduler → smoother convergence than a constant learning rate.
* paged_adamw_8bit → memory-efficient optimizer designed for QLoRA.
* fp16 → faster training and lower GPU memory usage on a Tesla T4.

### **Initialize the Supervised Fine-Tuning Trainer**

The SFTTrainer from the TRL library is used to fine-tune the TinyLlama model on the HospitalityGPT instruction dataset.

The trainer automatically tokenizes the formatted prompts and optimizes the LoRA parameters while keeping the base model frozen.

In [20]:
trainer = SFTTrainer(

    model=model,

    train_dataset=train_dataset,

    peft_config=peft_config,

    dataset_text_field="text",

    tokenizer=tokenizer,

    args=training_arguments,

    max_seq_length=512,

    packing=False

)

Map:   0%|          | 0/20184 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


### **Fine-Tune the Model**

The TinyLlama model is now fine-tuned on the HospitalityGPT dataset using Low-Rank Adaptation (LoRA).

During training, only the LoRA parameters are updated, while the original model weights remain frozen. This significantly reduces computational requirements while enabling effective domain adaptation.

In [21]:
trainer.train()

Step,Training Loss
25,2.209000
50,1.933400
75,1.325500
100,0.994200
125,0.816400
150,0.698700
175,0.621700
200,0.568800
225,0.526200
250,0.485200


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=5044, training_loss=0.28993081814896765, metrics={'train_runtime': 7885.2751, 'train_samples_per_second': 10.239, 'train_steps_per_second': 0.64, 'total_flos': 1.0984705448091648e+17, 'train_loss': 0.28993081814896765, 'epoch': 4.0})

### **Save the Fine-Tuned Model**

After training, the LoRA adapter, tokenizer, and training artifacts are saved to Google Drive.

These files will be loaded in the deployment notebook to generate hospitality-specific responses.

In [22]:
SAVE_PATH = f"{MODEL_PATH}/hospitalitygpt"

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved successfully to:\n{SAVE_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model saved successfully to:
/content/drive/MyDrive/Module_6_Specialization_NLP/HospitalityGPT/models/hospitalitygpt


### **Perform a Quick Inference Test**

A sample hospitality query is provided to the fine-tuned model to verify that the training process completed successfully and that the model is capable of generating domain-specific responses.

In [23]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.3
)

prompt = """You are HospitalityGPT, an AI assistant specialized in hotel and hospitality customer support.

### Instruction:
Can I request a late check-out?

### Input:

### Response:
"""

response = generator(prompt)[0]["generated_text"]

print(response)

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MvpForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCa

You are HospitalityGPT, an AI assistant specialized in hotel and hospitality customer support.

### Instruction:
Can I request a late check-out?

### Input:

### Response:
Yes, you can request a late check-out during your reservation or through our website, {{WEBSITE_URL}}. 1. Visit {{WEBSITE_HOTEL_SECTION}} and click on your reservation information. 2. Look for the {{CHECK_OUT_OPTION}} and select the late check-out request. 3. Compile your requirements and select the late check-out option. 4. Complete the request and await confirm


In [24]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.2,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


You are HospitalityGPT, an AI assistant specialized in hotel and hospitality customer support.

### Instruction:
Can I request a late check-out?

### Input:

### Response:
Yes, you can request a late check-out during your reservation or through our website at {{WEBSITE_URL}}. 1. Visit your account on {{WEBSITE_URL}}. 2. Go to the section where your booking information is displayed. 3. Locate the tab labeled {{CHECK_OUT_OPTION}}. 4. Choose the option for a late check-out and indicate your desired time. 5. Complete your request and await confirmation, which will be sent to you via email or through the app notification. If you face any difficulties or require prompt assistance, do not hesitate to reach out to our customer support team via the contact methods available on our site. We are committed to providing you with the best experience while you are staying with us.

PLEASE SHARE YOUR NEEDS IN THE COMPLETELY FILLED INFORMATION FIELD.

### Response:


## **Summary**

In this notebook, the TinyLlama 1.1B Chat model was successfully fine-tuned on the HospitalityGPT dataset using Low-Rank Adaptation (LoRA).

The following tasks were completed:

- Loaded the processed HospitalityGPT dataset
- Formatted instruction-response prompts
- Loaded the TinyLlama model with 4-bit quantization
- Configured LoRA for parameter-efficient fine-tuning
- Defined the training hyperparameters
- Fine-tuned the model using SFTTrainer
- Saved the fine-tuned model and tokenizer
- Verified the model using a sample inference

The resulting model is now ready to be integrated with a Retrieval-Augmented Generation (RAG) pipeline for improved factual accuracy and deployed as an interactive hospitality chatbot.